# CardioPath AI — EchoNext: EKG dan past EF (≤45%)

**Tartib:** har bir katakni yuqoridan pastga bittadan ishga tushiring (▶ tugmasi).

- Ma'lumot faqat shu Colab sessiyasida qoladi (PhysioNet shartnomasi: ma'lumotni boshqalarga tarqatish mumkin emas).
- Parol faqat so'ralganda yashirin maydonga yoziladi, faylga ham, ekranga ham chiqmaydi.
- Menyu: **Runtime → Change runtime type → T4 GPU** (o'qitish uchun).

## 1. Sinov: CSV (17 MB) ni yuklash
Agar `403` chiqsa, to'xtang va menga yozing.

In [ ]:
import getpass, os, subprocess, time
USER = 'umidjon001'
PW = getpass.getpass('PhysioNet paroli: ')
BASE = 'https://physionet.org/files/echonext/1.1.1/'
D = '/content/echonext'; os.makedirs(D, exist_ok=True)

def get(name):
    t = time.time()
    r = subprocess.run(['wget', '-q', '-c', '--user', USER, '--password', PW, '-O', f'{D}/{name}', BASE + name])
    size = os.path.getsize(f'{D}/{name}') if os.path.exists(f'{D}/{name}') else 0
    print(f'{name}: kod {r.returncode}, {size/1e6:.1f} MB, {time.time()-t:.0f} s, {size/1e6/max(1,time.time()-t):.1f} MB/s')
    return r.returncode == 0

ok = get('echonext_metadata_100k.csv')
print('ISHLADI' if ok else 'ISHLAMADI — parol yoki ruxsat (403). Menga yozing.')

## 2. Qolgan fayllar (val 1.0 GB, test 1.2 GB, train 16.2 GB) va tekshiruv

In [ ]:
for n in ['SHA256SUMS.txt', 'README.md', 'LICENSE.txt',
          'EchoNext_val_waveforms.npy', 'EchoNext_test_waveforms.npy', 'EchoNext_train_waveforms.npy',
          'EchoNext_val_tabular_features.npy', 'EchoNext_test_tabular_features.npy', 'EchoNext_train_tabular_features.npy']:
    get(n)
!cd /content/echonext && sha256sum -c SHA256SUMS.txt --ignore-missing
!df -h /content | tail -1

## 3. Ma'lumot tekshiruvi (o'qitishdan OLDIN)
Natijani (pastdagi chiqishni) to'liq nusxalab menga yuboring.

In [ ]:
import numpy as np, pandas as pd
m = pd.read_csv(f'{D}/echonext_metadata_100k.csv', index_col=0)
print('qatorlar:', len(m)); print(m['split'].value_counts())
for sp in ['train', 'val', 'test']:
    w = np.load(f'{D}/EchoNext_{sp}_waveforms.npy', mmap_mode='r')
    print(sp, 'waveforms', w.shape, w.dtype, '| metadata qatorlari', (m['split'] == sp).sum())
pk = {sp: set(m.loc[m.split == sp, 'patient_key']) for sp in ['train', 'val', 'test']}
print('bemor kesishmasi train∩val', len(pk['train'] & pk['val']), 'train∩test', len(pk['train'] & pk['test']), 'val∩test', len(pk['val'] & pk['test']))
y = m['lvef_lte_45_flag']
print('past EF ulushi (split bo`yicha):'); print(m.groupby('split')['lvef_lte_45_flag'].agg(['mean', 'count', lambda s: s.isna().sum()]))
for c in ['location_setting', 'sex', 'acquisition_year']:
    print(f'\npast EF ulushi — {c}:'); print(m[m.split != 'no_split'].groupby(c)['lvef_lte_45_flag'].agg(['mean', 'count']).round(3))
print('\nyosh (past EF 0/1):'); print(m.groupby('lvef_lte_45_flag')['age_at_ecg'].describe().round(1))